# Orbit Wars — Competitive Agent v6

A strong heuristic agent for the [Orbit Wars](https://www.kaggle.com/competitions/orbit-wars) Kaggle competition.

### Key improvements over baseline
- **Physics-based fleet tracking** — simulates each fleet's trajectory to determine exactly which planet it will hit, not just angle-cone guessing
- **Iterative orbit intercept** — solves for the moving planet's future position accounting for fleet travel time
- **Overwhelming force doctrine** — sends 150% of garrison + buffer, never wastes ships on losing attacks  
- **Multi-planet coordinated assaults** — combines forces from multiple planets against strong targets
- **4-player awareness** — prioritizes weaker opponents in multiplayer games
- **Sun-safe pathfinding** — avoids fleet-destroying sun crossings


In [1]:
# Step 1: Upgrade kaggle-environments (run once, then restart kernel)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle-environments"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Upgrade done! Now: Kernel → Restart, then skip this cell and run from next cell.")
else:
    print("⚠️", result.stderr[-300:])


✅ Upgrade done! Now: Kernel → Restart, then skip this cell and run from next cell.


In [2]:
%%writefile submission.py
import math
from collections import namedtuple, defaultdict

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet  = namedtuple("Fleet",  ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

BOARD = 100.0
CX = 50.0
CY = 50.0
SUN_R = 10.0
ROT_LIM = 50.0
MAX_SPEED = 6.0
STEPS = 500

# ─────────────────────────────────────────────────────────────────
# GEOMETRY
# ─────────────────────────────────────────────────────────────────
def dist(x1, y1, x2, y2):
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)

def fleet_speed(n):
    if n <= 1: return 1.0
    return min(MAX_SPEED, 1.0 + (MAX_SPEED-1.0) * (math.log(max(n,2))/math.log(1000))**1.5)

def eta(d, n):
    s = fleet_speed(n)
    return d / s if s > 0 else 9999

def predict_pos(init, av, step):
    """Predict planet position at given step. init = [id,owner,x,y,r,ships,prod]"""
    dx, dy = init[2]-CX, init[3]-CY
    orb = math.sqrt(dx*dx + dy*dy)
    if orb + init[4] >= ROT_LIM:
        return init[2], init[3]
    a0 = math.atan2(dy, dx)
    a = a0 + av * step
    return CX + orb*math.cos(a), CY + orb*math.sin(a)

def intercept(sx, sy, init, av, step, n):
    """Find where to aim to hit an orbiting planet with n ships from (sx,sy)."""
    tx, ty = predict_pos(init, av, step+3)
    for _ in range(12):
        d = dist(sx, sy, tx, ty)
        t = eta(d, n)
        fs = step + max(1, int(t+0.5))
        tx, ty = predict_pos(init, av, fs)
    return tx, ty, eta(dist(sx, sy, tx, ty), n)

def sun_blocked(sx, sy, tx, ty):
    """Check if path from s to t crosses sun (with margin)."""
    dx, dy = tx-sx, ty-sy
    l2 = dx*dx + dy*dy
    if l2 < 0.01: return False
    t = max(0, min(1, ((CX-sx)*dx + (CY-sy)*dy) / l2))
    px, py = sx+t*dx, sy+t*dy
    return dist(px, py, CX, CY) < SUN_R + 2.5

def safe_angle(sx, sy, tx, ty, sr):
    """Get launch angle avoiding sun. Returns None if impossible."""
    direct = math.atan2(ty-sy, tx-sx)
    lx = sx + math.cos(direct)*(sr+0.2)
    ly = sy + math.sin(direct)*(sr+0.2)
    if not sun_blocked(lx, ly, tx, ty):
        return direct
    for nudge in [0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5]:
        for sign in [1, -1]:
            a = direct + sign*nudge
            lx2 = sx + math.cos(a)*(sr+0.2)
            ly2 = sy + math.sin(a)*(sr+0.2)
            if not sun_blocked(lx2, ly2, tx, ty):
                return a
    return None

def will_hit_planet(fx, fy, angle, fships, px, py, pr):
    """Simulate if fleet from (fx,fy) at angle will collide with planet at (px,py,pr)."""
    spd = fleet_speed(fships)
    for s in range(1, 60):
        nx = fx + math.cos(angle)*spd*s
        ny = fy + math.sin(angle)*spd*s
        if nx < -5 or nx > 105 or ny < -5 or ny > 105:
            return False, 9999
        if dist(nx, ny, px, py) < pr + 0.5:
            return True, s
    return False, 9999

# ─────────────────────────────────────────────────────────────────
# FLEET ANALYSIS
# ─────────────────────────────────────────────────────────────────
def analyze_fleets(planets, fleets, player):
    """Track ships heading toward each planet."""
    my_incoming = defaultdict(int)     # planet_id -> my ships heading there
    enemy_incoming = defaultdict(int)  # planet_id -> enemy ships heading to my planets
    my_incoming_eta = defaultdict(lambda: 9999)
    
    planet_map = {p.id: p for p in planets}
    
    for f in fleets:
        # Find which planet this fleet will likely hit
        best_pid = None
        best_arrive = 9999
        
        for p in planets:
            hit, arrive = will_hit_planet(f.x, f.y, f.angle, f.ships, p.x, p.y, p.radius)
            if hit and arrive < best_arrive:
                best_arrive = arrive
                best_pid = p.id
        
        if best_pid is None:
            continue
            
        if f.owner == player:
            my_incoming[best_pid] += f.ships
            my_incoming_eta[best_pid] = min(my_incoming_eta[best_pid], best_arrive)
        else:
            target = planet_map.get(best_pid)
            if target and target.owner == player:
                enemy_incoming[best_pid] += f.ships
    
    return my_incoming, enemy_incoming, my_incoming_eta

# ─────────────────────────────────────────────────────────────────
# MAIN AGENT
# ─────────────────────────────────────────────────────────────────
def agent(obs, config=None):
    moves = []
    player = obs.get("player", 0)
    step = obs.get("step", 0)
    av = obs.get("angular_velocity", 0)
    
    planets = [Planet(*p) for p in obs.get("planets", [])]
    fleets  = [Fleet(*f)  for f in obs.get("fleets", [])]
    comet_ids = set(obs.get("comet_planet_ids", []))
    
    init_map = {}
    for p in obs.get("initial_planets", []):
        init_map[p[0]] = p
    
    my_p = [p for p in planets if p.owner == player]
    enemy_p = [p for p in planets if p.owner != player and p.owner != -1]
    neutral_p = [p for p in planets if p.owner == -1]
    targets = [p for p in planets if p.owner != player]
    
    if not my_p:
        return []
    
    num_players = len(set(p.owner for p in planets if p.owner >= 0) | set(f.owner for f in fleets))
    
    # Analyze fleet trajectories
    my_inc, enemy_inc, my_inc_eta = analyze_fleets(planets, fleets, player)
    
    # Budget: available ships per planet (defense-reserved)
    budget = {}
    for p in my_p:
        threat = enemy_inc.get(p.id, 0)
        if threat > 0:
            reserve = min(p.ships, int(threat * 1.3) + 5)
        else:
            reserve = 0
        budget[p.id] = max(0, p.ships - reserve)
    
    committed = set()  # target planet IDs we've already assigned an attack to
    
    # ═════════════════════════════════════════════════════════════
    # SCORE ALL ATTACKS
    # ═════════════════════════════════════════════════════════════
    attacks = []
    
    for tgt in targets:
        is_comet = tgt.id in comet_ids
        already = my_inc.get(tgt.id, 0)
        
        for src in my_p:
            avail = budget.get(src.id, 0)
            if avail < 3:
                continue
            
            # Get intercept position
            if tgt.id in init_map and not is_comet:
                tx, ty, t = intercept(src.x, src.y, init_map[tgt.id], av, step, max(avail//2, 5))
            else:
                tx, ty = tgt.x, tgt.y
                t = eta(dist(src.x, src.y, tx, ty), max(avail//2, 5))
            
            if step + t > STEPS - 2:
                continue
            
            d = dist(src.x, src.y, tx, ty)
            if d < 1.5:
                continue
            
            # Garrison at arrival
            garrison = tgt.ships
            if tgt.owner != -1:
                garrison += int(tgt.production * t)
            
            # Subtract what we already have heading there
            net = max(0, garrison - already)
            
            # If we've already sent enough, skip
            if already > garrison * 1.3 + 5:
                continue
            
            # Ships to send: OVERWHELMING force
            need = int(net * 1.5) + 4
            if need < 4:
                need = 4
            
            if avail < need:
                continue
            
            # Will we actually win?
            if need <= net:
                continue
            
            # VALUE
            remain = max(1, STEPS - step - t)
            val = tgt.production * remain
            if tgt.owner != -1:
                val += tgt.production * remain * 0.6  # deny enemy production
            if is_comet:
                val *= 0.15
            
            # Efficiency
            score = val / max(need, 1)
            score -= d * 0.08  # distance penalty
            
            # Early game: HUGE bonus for quick expansion
            if step < 80:
                if tgt.owner == -1:
                    if d < 20:
                        score *= 3.0
                    elif d < 35:
                        score *= 2.0
                    else:
                        score *= 1.3
            
            # In 4-player: prioritize weakest enemy
            if num_players > 2 and tgt.owner != -1:
                enemy_str = sum(p.ships for p in planets if p.owner == tgt.owner)
                my_str = sum(p.ships for p in my_p)
                if enemy_str < my_str * 0.5:
                    score *= 1.5  # go after weak opponents
            
            attacks.append({
                "tid": tgt.id,
                "src_id": src.id,
                "src": src,
                "need": need,
                "tx": tx, "ty": ty,
                "score": score,
                "net": net,
                "eta": t,
            })
    
    attacks.sort(key=lambda a: a["score"], reverse=True)
    
    # ═════════════════════════════════════════════════════════════
    # EXECUTE TOP ATTACKS
    # ═════════════════════════════════════════════════════════════
    for atk in attacks:
        tid = atk["tid"]
        sid = atk["src_id"]
        
        if tid in committed:
            continue
        
        avail = budget.get(sid, 0)
        need = atk["need"]
        
        if avail < need:
            continue
        
        # Keep minimum garrison on source
        min_keep = max(1, atk["src"].production)
        if step < 5:
            min_keep = 0
        
        send = min(need, avail - min_keep)
        if send < 3:
            continue
        
        # Final check: will we actually win combat?
        if send <= atk["net"]:
            continue
        
        angle = safe_angle(atk["src"].x, atk["src"].y, atk["tx"], atk["ty"], atk["src"].radius)
        if angle is None:
            continue
        
        moves.append([sid, angle, send])
        budget[sid] -= send
        committed.add(tid)
    
    # ═════════════════════════════════════════════════════════════
    # MULTI-SOURCE ATTACKS on strong enemy planets
    # ═════════════════════════════════════════════════════════════
    if step > 30:
        for tgt in enemy_p:
            if tgt.id in committed:
                continue
            
            already = my_inc.get(tgt.id, 0)
            garrison = tgt.ships
            need_total = int(garrison * 1.5) + 8 - already
            
            if need_total < 10:
                continue
            
            # Gather potential senders
            senders = []
            for src in my_p:
                av_s = budget.get(src.id, 0)
                if av_s < 5:
                    continue
                d = dist(src.x, src.y, tgt.x, tgt.y)
                if d > 55:
                    continue
                senders.append((d, src, av_s))
            
            total_avail = sum(a for _, _, a in senders)
            if total_avail < need_total:
                continue
            
            senders.sort()  # closest first
            sent = 0
            for d, src, av_s in senders:
                if sent >= need_total:
                    break
                to_send = min(av_s - max(1, src.production), need_total - sent)
                if to_send < 5:
                    continue
                
                if tgt.id in init_map and tgt.id not in comet_ids:
                    tx, ty, _ = intercept(src.x, src.y, init_map[tgt.id], av, step, to_send)
                else:
                    tx, ty = tgt.x, tgt.y
                
                angle = safe_angle(src.x, src.y, tx, ty, src.radius)
                if angle is None:
                    continue
                
                moves.append([src.id, angle, to_send])
                budget[src.id] -= to_send
                sent += to_send
            
            if sent > 0:
                committed.add(tgt.id)
    
    # ═════════════════════════════════════════════════════════════
    # REINFORCEMENT: push excess ships toward front line
    # ═════════════════════════════════════════════════════════════
    if step > 20 and enemy_p:
        for src in my_p:
            avail = budget.get(src.id, 0)
            if avail < 20:
                continue
            
            src_threat = min(dist(src.x, src.y, e.x, e.y) for e in enemy_p)
            
            best_dst = None
            best_v = 0
            for dst in my_p:
                if dst.id == src.id:
                    continue
                dst_threat = min(dist(dst.x, dst.y, e.x, e.y) for e in enemy_p)
                if dst_threat >= src_threat - 3:
                    continue
                d = dist(src.x, src.y, dst.x, dst.y)
                if d < 3 or d > 45:
                    continue
                v = (src_threat - dst_threat) / max(d, 1)
                if v > best_v:
                    best_v = v
                    best_dst = dst
            
            if best_dst and best_v > 0.05:
                send = avail // 2
                if send >= 10:
                    angle = safe_angle(src.x, src.y, best_dst.x, best_dst.y, src.radius)
                    if angle is not None:
                        moves.append([src.id, angle, send])
                        budget[src.id] -= send
    
    # ═════════════════════════════════════════════════════════════
    # LATE GAME: throw everything at beatable enemies
    # ═════════════════════════════════════════════════════════════
    if step > 380:
        for src in my_p:
            avail = budget.get(src.id, 0)
            if avail < 10:
                continue
            
            best, best_r = None, 0
            for ep in enemy_p:
                if ep.id in committed:
                    continue
                d = dist(src.x, src.y, ep.x, ep.y)
                t = eta(d, avail)
                if step + t > STEPS - 2:
                    continue
                g = ep.ships + int(ep.production * t)
                if avail > g + 3:
                    r = avail / max(g, 1)
                    if r > best_r:
                        best_r, best = r, ep
            
            if best and best_r > 1.2:
                if best.id in init_map:
                    tx, ty, _ = intercept(src.x, src.y, init_map[best.id], av, step, avail)
                else:
                    tx, ty = best.x, best.y
                angle = safe_angle(src.x, src.y, tx, ty, src.radius)
                if angle is not None:
                    send = avail - 1
                    moves.append([src.id, angle, send])
                    budget[src.id] = 1
    
    return moves


Writing submission.py


## Testing

> **If you get `Unknown Environment Specification`:** Run the pip upgrade cell above, then **restart the kernel**, then skip that cell and run from the `%%writefile` cell onward.


In [3]:
from kaggle_environments import make

results = {"W": 0, "L": 0}
for i in range(5):
    env = make("orbit_wars", debug=False)
    env.run(["submission.py", "starter"])
    r = env.steps[-1][0]["reward"]
    key = "W" if r == 1 else "L"
    results[key] += 1
    obs = env.steps[-1][0]["observation"]
    s0 = sum(p[5] for p in obs.get("planets", []) if p[1] == 0)
    s0 += sum(f[6] for f in obs.get("fleets", []) if f[1] == 0)
    s1 = sum(p[5] for p in obs.get("planets", []) if p[1] == 1)
    s1 += sum(f[6] for f in obs.get("fleets", []) if f[1] == 1)
    print(f"Game {i+1}: {s0:5d} vs {s1:5d} -> {key}")

print(f"\nVs Starter: {results['W']}W / {results['L']}L")


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

In [4]:
# # Replay visualization (optional)
# env = make("orbit_wars", debug=True)
# env.run(["submission.py", "starter"])
# env.render(mode="ipython", width=800, height=800)
